In [26]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

openai.api_key  = os.environ['OPENROUTER_API_KEY']

In [27]:

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
  api_key=os.environ['OPENROUTER_API_KEY'],
  base_url="https://openrouter.ai/api/v1",
  model="openrouter/free",
  temperature=0
)

# 1.Data Loading

In [28]:
from langchain_community.document_loaders import PyPDFLoader
# making the file path relative to the current file
Data_path = "../data/HamzaEldafrawy.pdf"
loader = PyPDFLoader(Data_path)
pages = loader.load()

C:\Users\Hamza Eldafrawy\AppData\Local\Temp\ipykernel_27344\1583520230.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
len(pages)

2

In [ ]:
print(pages[0].page_content)

Hamza Basem Mohamed Ahmed Eldafrawy
Cairo, Egypt — 01140728368 — hamza.eldefrawy4@gmail.com
LinkedIn — GitHub
PROFILE
Computer and Systems Engineering student (Class of 2027, GPA: 3.62) with experience in backend development, distributed
systems, machine learning, and data-driven architectures. Built foundational projects involving socket programming, recom-
mendation systems, and software architecture using Python, C#, Java, and C++. Dedicated to a first-principles engineering
mindset, combining robust backend engineering with intelligent systems to build scalable, real-world applications.
EDUCATION
Ain Shams University — Faculty of EngineeringCairo, Egypt
B.Sc. in Computer and Systems EngineeringExpected Graduation: 2027
•Cumulative GPA:3.62/4.00
•Relevant Coursework:Deep Learning, Data Structures & Algorithms, Operating Systems, Parallel and Distributed Systems,
Databases, Fundamentals of Big Data Analysis.
TECHNICAL SKILLS
•Programming Languages:Python, C#, SQL, C++, C, Java
•Data 

## Adding Data from Github

In [45]:
import os
import json
import base64
from pathlib import Path
from urllib.parse import quote

import requests
from dotenv import load_dotenv


load_dotenv()

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

token = os.getenv("GITHUB_TOKEN")
if token:
    session.headers["Authorization"] = f"Bearer {token}"


def get_json(url, params=None):
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def fetch_github_profile(username: str) -> dict:
    return get_json(f"https://api.github.com/users/{username}")


def fetch_github_repositories(username: str) -> list[dict]:
    url = f"https://api.github.com/users/{username}/repos"
    repositories = []
    page = 1

    while True:
        batch = get_json(
            url,
            params={
                "per_page": 100,
                "page": page,
                "sort": "updated",
                "type": "owner",
            },
        )

        if not batch:
            break

        repositories.extend(batch)
        page += 1

    return repositories


def fetch_repository_content(
    owner: str,
    repo: dict,
    max_files: int = 30,
    max_file_size: int = 100_000,
) -> dict:
    """Fetch repository metadata, README, and selected source files."""

    name = repo["name"]
    base_url = f"https://api.github.com/repos/{owner}/{name}"

    result = {
        "metadata": repo,
        "readme": None,
        "source_files": [],
    }

    # README
    readme_response = session.get(
        f"{base_url}/readme",
        timeout=30,
    )

    if readme_response.status_code == 200:
        readme = readme_response.json()
        result["readme"] = {
            "path": readme["path"],
            "url": readme["html_url"],
            "content": base64.b64decode(
                readme["content"]
            ).decode("utf-8"),
        }
    elif readme_response.status_code != 404:
        readme_response.raise_for_status()

    # Repository file tree
    branch = repo["default_branch"]
    tree_url = (
        f"{base_url}/git/trees/{quote(branch, safe='')}"
    )
    tree_data = get_json(tree_url, {"recursive": "1"})

    if tree_data.get("truncated"):
        print(f"Warning: GitHub returned a truncated tree for {name}")

    extensions = {
        ".py", ".cs", ".java", ".cpp", ".c", ".h", ".hpp",
        ".js", ".ts", ".sql", ".sh", ".ipynb", ".md", ".yaml", ".yml",
    }

    excluded_dirs = {
        ".git", ".venv", "venv", "node_modules",
        "__pycache__", "dist", "build",
    }

    excluded_files = {
        ".env", "secrets.json", "credentials.json",
    }

    for item in tree_data["tree"]:
        if len(result["source_files"]) >= max_files:
            break

        if item["type"] != "blob":
            continue

        path = item["path"]
        parts = path.split("/")
        filename = parts[-1]

        extension = (
            "." + filename.rsplit(".", 1)[-1].lower()
            if "." in filename else ""
        )

        if any(part.lower() in excluded_dirs for part in parts):
            continue

        if filename.lower() in excluded_files:
            continue

        if extension not in extensions:
            continue

        if item.get("size", 0) > max_file_size:
            continue

        raw_url = (
            f"https://raw.githubusercontent.com/{owner}/{name}/"
            f"{quote(branch, safe='')}/{quote(path, safe='/')}"
        )

        try:
            response = session.get(raw_url, timeout=30)
            response.raise_for_status()
            content = response.content.decode("utf-8")
        except (requests.RequestException, UnicodeDecodeError):
            continue

        if not content.strip():
            continue

        result["source_files"].append({
            "path": path,
            "url": (
                f"{repo['html_url']}/blob/{quote(branch, safe='')}/"
                f"{quote(path, safe='/')}"
            ),
            "content": content,
        })

    return result


def ingest_github(
    username: str,
    output_dir: str = "../data/github",
    max_files_per_repo: int = 30,
    force_fetch: bool = False,
) -> Path:
    """Save a GitHub profile and its owned public repositories locally."""

    root = Path(output_dir) / username
    root.mkdir(parents=True, exist_ok=True)

    # Save profile
    
    profile_path = root / "profile.json"

    if profile_path.exists():
        print("Profile already exists; skipping download.")
    else:
        profile = fetch_github_profile(username)
        save_json(profile_path, profile)

    # Retrieve repositories
    repositories = fetch_github_repositories(username)
    repos_dir = root / "repositories"
    repos_dir.mkdir(parents=True, exist_ok=True)

    # Fetch and save each repository
    for index, repo in enumerate(repositories, start=1):
        repo_path = repos_dir / f"{repo['name']}.json"

        if repo_path.exists() and not force_fetch:
            print(f"[{index}/{len(repositories)}] Skipping {repo['name']} (already exists)")
            continue
        try:
            data = fetch_repository_content(
                username,
                repo,
                max_files=max_files_per_repo,
            )

            save_json(
                repos_dir / f"{repo['name']}.json",
                data,
            )

            print(f"[{index}/{len(repositories)}] Saved {repo['name']}")

        except requests.RequestException as error:
            print(f"Failed to fetch {repo['name']}: {error}")

    print(f"GitHub ingestion finished: {root.resolve()}")
    return root

In [46]:
data_path = ingest_github("Hamza-Eldefrawy", force_fetch=True)

Profile already exists; skipping download.
[1/22] Saved personal-rag-system
[2/22] Saved Toxic-Text-Classifier
[3/22] Saved distributed-marketplace
[4/22] Saved vezeeta-pulse
[5/22] Saved Algorithm-Design-Project
[6/22] Saved mlops-practitioner
[7/22] Saved OS-Memory-Allocator
[8/22] Saved BankManagementSystem
[9/22] Saved Data_Analytics
[10/22] Saved alx_be_python
[11/22] Saved GitHubCloneProject
[12/22] Saved LocalRepoProject
[13/22] Saved Elevvo-Internship
[14/22] Saved Online-Language-Teaching-System
[15/22] Saved GPS_Project_Embedded
[16/22] Saved Unix_Utilities
[17/22] Saved Digital-Design-Project
[18/22] Saved alx-low_level_programming
[19/22] Saved zero_day
[20/22] Saved alx-system_engineering-devops
[21/22] Saved alx-zero_day
[22/22] Saved alx-pre_course
GitHub ingestion finished: C:\Users\Hamza Eldafrawy\Desktop\Cellula_Intern\personal-RAG\data\github\Hamza-Eldefrawy


# 2.Chunking

In [23]:
import json
from pathlib import Path
from langchain_core.documents import Document


def load_saved_github_documents(
    username: str,
    data_dir: str = "../data/github",
) -> list[Document]:
    """Load saved GitHub JSON data into LangChain Documents."""

    root = Path(data_dir) / username
    documents = []

    # 1. Load profile
    profile_path = root / "profile.json"

    if profile_path.exists():
        profile = json.loads(profile_path.read_text(encoding="utf-8"))

        profile_text = "\n".join(
            f"{key}: {value}"
            for key, value in profile.items()
            if value is not None
        )

        documents.append(
            Document(
                page_content=profile_text,
                metadata={
                    "source": profile.get("html_url", ""),
                    "type": "github_profile",
                    "username": profile.get("login", username),
                },
            )
        )

    # 2. Load repositories
    repos_dir = root / "repositories"

    if repos_dir.exists():
        for repo_path in repos_dir.glob("*.json"):
            data = json.loads(repo_path.read_text(encoding="utf-8"))
            repo = data["metadata"]

            repo_name = repo.get("full_name", repo_path.stem)
            repo_url = repo.get("html_url", "")

            # Repository metadata
            metadata_text = "\n".join([
                f"Repository: {repo_name}",
                f"Description: {repo.get('description') or 'N/A'}",
                f"Language: {repo.get('language') or 'N/A'}",
                f"Topics: {', '.join(repo.get('topics') or [])}",
                f"Stars: {repo.get('stargazers_count', 0)}",
                f"URL: {repo_url}",
            ])

            documents.append(
                Document(
                    page_content=metadata_text,
                    metadata={
                        "source": repo_url,
                        "type": "repository_metadata",
                        "repository": repo_name,
                    },
                )
            )

            # README
            readme = data.get("readme")

            if readme and readme.get("content"):
                documents.append(
                    Document(
                        page_content=readme["content"],
                        metadata={
                            "source": readme.get("url", repo_url),
                            "type": "readme",
                            "repository": repo_name,
                            "file_path": readme.get("path", "README"),
                        },
                    )
                )

            # Source files
            for file in data.get("source_files", []):
                if not file.get("content"):
                    continue

                documents.append(
                    Document(
                        page_content=file["content"],
                        metadata={
                            "source": file.get("url", repo_url),
                            "type": "source_code",
                            "repository": repo_name,
                            "file_path": file.get("path", ""),
                        },
                    )
                )

    return documents

In [24]:
documents = load_saved_github_documents("Hamza-Eldefrawy")

In [15]:
from langchain_classic.text_splitter import TokenTextSplitter
splitter = TokenTextSplitter(
  chunk_size = 200,
  chunk_overlap = 20
)


In [29]:
chunks = splitter.split_documents(pages)
chunks += splitter.split_documents(documents)

In [30]:
len(chunks)

1223

# 3.Embedding and Vector DB

In [31]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-MiniLM-L6-v2",
  model_kwargs = {"device":"cpu"}
)
vectors = embeddings.embed_documents([i.page_content for i in chunks])
# vector store
from langchain_classic.vectorstores import FAISS
vectorDB = FAISS.from_documents(chunks,embeddings)

C:\Users\Hamza Eldafrawy\AppData\Local\Temp\ipykernel_27344\2271255690.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Hamza Eldafrawy\Desktop\Cellula_Intern\personal-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 440.83it/s]


# 4.Retrieval

In [32]:
query = "What is the personal information of Hamza Eldafrawy?"
similar_docs = vectorDB.similarity_search_with_score(query,k=3)
for i in similar_docs :
  print(i[0].page_content)

: https://api.github.com/users/Hamza-Eldefrawy/received_events
type: User
user_view_type: public
site_admin: False
name: Hamza Eldafrawy
blog: 
public_repos: 22
public_gists: 0
followers: 6
following: 10
created_at: 2020-10-04T11:44:11Z
updated_at: 2026-07-15T01:25:05Z
login: Hamza-Eldefrawy
id: 72341927
node_id: MDQ6VXNlcjcyMzQxOTI3
avatar_url: https://avatars.githubusercontent.com/u/72341927?v=4
gravatar_id: 
url: https://api.github.com/users/Hamza-Eldefrawy
html_url: https://github.com/Hamza-Eldefrawy
followers_url: https://api.github.com/users/Hamza-Eldefrawy/followers
following_url: https://api.github.com/users/Hamza-Eldefrawy/following{/other_user}
gists_url: https://api.github.com/users/Hamza-Eldefrawy/gists{
api.github.com/users/Hamza-Eldefrawy/gists{/gist_id}
starred_url: https://api.github.com/users/Hamza-Eldefrawy/starred{/owner}{/repo}
subscriptions_url: https://api.github.com/users/Hamza-Eldefrawy/subscriptions
organizations_url: https://api.github.com/users/Hamza-Eldefraw

# 5.Generation

In [33]:
from langchain_classic.prompts import PromptTemplate

text = """
You are a helpful assistant that answers questions about a person.
If you don't know the answer, say you don't know.
Keep your answer within three sentences.Keep the answer as concise as possible.
Use the following pieces of context to answer the question at the end. 
Always say "Any further questions?" at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
template = PromptTemplate.from_template(text)

In [41]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

chat = ConversationalRetrievalChain.from_llm(
  llm = llm,
  retriever = vectorDB.as_retriever(),
  return_source_documents = True,
  combine_docs_chain_kwargs={"prompt": template},
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
)

In [44]:
response = chat.invoke({
    "question": "what about RAG?",
})

print(response["answer"])
# print(response["source_documents"])


The context contains a repository named `personal-rag-system` but lacks details about Hamza's RAG experience. I don't know based on the given information. Any further questions?
